In [ ]:
# Install packages
%pip install langchain-deepseek
%pip install deepagents tavily-python

In [ ]:
# Import packages
from langchain.messages import AIMessage
from langchain_core.output_parsers import PydanticOutputParser
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage
import os
from google.colab import userdata
from langchain.tools import tool


In [ ]:
# Connect notebook to langsmith
os.environ['LANGSMITH_TRACING'] = userdata.get('LANGSMITH_TRACING')
os.environ['LANGSMITH_ENDPOINT'] = userdata.get('LANGSMITH_ENDPOINT')
os.environ['LANGSMITH_API_KEY'] = userdata.get('LANGSMITH_API_KEY')
os.environ['LANGSMITH_PROJECT'] = userdata.get('LANGSMITH_PROJECT')
os.environ['DEEPSEEK_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')
os.environ['TAVILY_API_KEY'] = userdata.get('TAVILY')
# Test the output
userdata.get('LANGSMITH_PROJECT')

In [ ]:
# Create a search tool
import os
from typing import Literal

from tavily import TavilyClient
from deepagents import create_deep_agent

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

In [ ]:
# System prompt to steer the agent to be an expert researcher
research_instructions = """You are an expert researcher.
Your job is to conduct thorough research and then write an overview.

You have access to an internet search tool as your primary means of gathering information.

## `internet_search`

Use this to run an internet search for a given query. You can specify the max number of results to return, the topic, and whether raw content should be included.
"""

agent = create_deep_agent(
    model="deepseek:deepseek-v4-flash",
    tools=[internet_search],
    system_prompt=research_instructions,
)

In [ ]:
for message in agent.stream_events({"messages": [{"role": "user", "content": "What's been happening in the news recently?"}],}, version="v3").messages:
    for delta in message.text:
        print(delta, end="\n", flush=True)

final_state = agent.stream_events({
    "messages": [{"role": "user", "content": "What's been happening in the news recently?"}],
}, version="v3").output